# Vehicle Type Recognition from Images
## Part 1 — Data Audit and Leakage-Controlled Partitioning

**Dataset:** VTID2 (Vehicle Type Image Dataset 2)
**Author:** Mohammed Mahyoub Ali Ayedh Mohammed — MSc Artificial Intelligence, UWE Bristol

This notebook produces the partitioned dataset used by `02_cnn_modelling.ipynb`.
It exists to answer a single question before any model is trained:

> Can an accuracy measured on this dataset be believed?

# 1. Purpose and Scope

A convolutional network is evaluated on the partition it is given. If the test
partition contains images that share a source with training images, the reported
accuracy measures memorisation of specific photographs rather than recognition of
vehicle types. Barz and Denzler (2020) found that 3.3% of the CIFAR-100 test set
duplicates training images and that removing them changes the measured ranking of
published models. The risk is not hypothetical, and it is larger in a dataset such
as VTID2, where images of the same vehicle may be captured seconds apart and where
some copies have been modified for anonymisation.

## 1.1 Two decisions, deliberately separated

The audit makes two different decisions about a pair of similar images, and the
central design choice of this notebook is to keep them apart:

| | Deletion | Partition constraint |
|---|---|---|
| Action | Remove one image permanently | Force both images into the same partition |
| If the decision is wrong | Training data is destroyed and cannot be recovered | A few images sit in one partition rather than another |
| Evidence required | Near-certainty | Reasonable suspicion |
| Optimisation target | **Precision** | **Recall** |

These two targets are in direct conflict, so no single similarity threshold can
serve both. An earlier exploratory version of this analysis searched for one, and
the search did not terminate: each candidate threshold was tested by inspecting
grids of image pairs, every grid produced counter-examples in both directions, and
the notebook grew without converging. The problem was not the threshold. It was
asking one number to answer two questions with opposite cost structures.

Separating them makes both decisions tractable:

- **Deletion** uses a rule calibrated to make *no* false deletions on a manually
  labelled reference set, and applies only to groups whose every internal pair was
  independently accepted.
- **Partitioning** uses a deliberately permissive relation, keeps every suspicious
  image together, and deletes nothing.

## 1.2 What this notebook produces

1. `results/inventory.csv` — one audited record per source file.
2. `results/split_manifest.csv` — the final image list with class, group and partition.
3. `results/audit_summary.csv` — the summary table reproduced in the report.
4. `figures/` — the retained visual evidence, stored as files rather than embedded.

# 2. Method

The audit runs as four levels. Each level answers a narrower question than the one
before it and is applied only to what the previous level could not resolve, so the
expensive comparisons are never run over the whole dataset.

| Level | Question answered | Instrument | Applied to |
|---|---|---|---|
| 1. File integrity | Can every file be decoded, and do its format or dimensions leak class information? | Full decode and metadata | All files |
| 2. Exact duplication | Is this literally the same file? | SHA-256 of the raw bytes | All files |
| 3. Candidate generation | *Could* these two images share a source? | Cosine similarity of the ResNet-18 global embedding | All pairs |
| 4. Identity decision | *Do* these two images share a source? | Spatial agreement and difference localisation over `layer2` and `layer3` feature maps | Candidate pairs only |

Level 3 is a *blocking* stage in the sense used in record linkage (Christen, 2012):
it is tuned for recall, because a pair it discards can never be recovered by Level 4.
Level 4 is the decision stage and is tuned for precision.

## 2.1 Why the levels are ordered this way

Levels 3 and 4 use the same pretrained network but read it at different depths, and
the distinction is the technical core of this notebook.

The final convolutional block of a residual network encodes *what* an image
contains. After global average pooling, all spatial arrangement has been discarded,
so two photographs of two different silver pickup trucks in similar poses receive
nearly the same 512-dimensional vector. That representation answers "are these the
same kind of thing?", which is not the question the audit asks.

The intermediate blocks retain a spatial grid. Each location in a `layer2` or
`layer3` feature map describes the mid-level structure — edges, corners, textures,
part boundaries — present at a specific position in the image. Two images that
share a source agree *location by location*; two different photographs of similar
vehicles do not, because their content is not registered to the same grid. This
follows the standard account of hierarchical convolutional representation (Zeiler
and Fergus, 2014).

Section 9 tests this claim quantitatively rather than assuming it.

# 3. Environment and Configuration

Library versions are pinned in `requirements.txt`. Installation and driver issues
encountered while preparing the environment are not reproduced here: they are not
findings about the data, and a report is not a laboratory diary.

All constants live in `src/config.py` so that the analysis has one auditable source
of truth, and all mechanical operations live in `src/vtid_audit.py` so that this
notebook contains only the reasoning specific to the study.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

import config as cfg
import vtid_audit as va

va.set_seed(cfg.RANDOM_SEED)

for directory in (cfg.CACHE_DIR, cfg.LABEL_DIR, cfg.FIGURE_DIR, cfg.RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch       : {torch.__version__}")
print(f"Torchvision   : {torchvision.__version__}")
print(f"Device        : {device}")
if device.type == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"Dataset root  : {cfg.DATA_ROOT}")
print(f"Random seed   : {cfg.RANDOM_SEED}")

# 4. Level 1 — Inventory and File Integrity

Every file is fully decoded rather than merely opened, so truncated images are
detected here instead of interrupting a training run hours later. Alongside
integrity, the inventory records the properties that could allow a network to
identify a class without looking at the vehicle: file format, colour mode,
resolution and aspect ratio.

In [ ]:
inventory = va.build_inventory(cfg.DATA_ROOT)
inventory.to_csv(cfg.RESULT_DIR / "inventory.csv", index=False)

va.integrity_report(inventory)

In [ ]:
class_counts = (
    inventory["class_name"]
    .value_counts()
    .reindex(cfg.CLASS_ORDER)
    .rename("Images")
    .to_frame()
)
class_counts["Share (%)"] = (100 * class_counts["Images"] / len(inventory)).round(2)
class_counts.index = [cfg.CLASS_DISPLAY[name] for name in class_counts.index]

class_counts

## 4.1 Resolution as a possible shortcut

If one class were systematically stored at a different resolution from the others,
a convolutional network could separate the classes from the resampling artefacts
alone. The check below compares the distribution of image dimensions across classes;
it is a test for a confound, not a description of the images.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))

class_counts["Images"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Images per class")
axes[0].set_ylabel("Images")
axes[0].tick_params(axis="x", rotation=0)

readable = inventory[inventory["readable"]]
for name in cfg.CLASS_ORDER:
    subset = readable[readable["class_name"] == name]
    axes[1].scatter(
        subset["width"], subset["height"], s=6, alpha=0.35,
        label=cfg.CLASS_DISPLAY[name],
    )
axes[1].set_title("Stored resolution by class")
axes[1].set_xlabel("Width (px)")
axes[1].set_ylabel("Height (px)")
axes[1].legend(fontsize=7, markerscale=2)

figure.tight_layout()
va.save_figure(figure, "01_class_balance_and_resolution", cfg.FIGURE_DIR)
plt.show()

In [ ]:
resolution = readable.assign(
    resolution=readable["width"].astype(int).astype(str)
    + "x"
    + readable["height"].astype(int).astype(str)
)

resolution_by_class = (
    resolution.groupby("class_name")
    .agg(
        Images=("image_id", "size"),
        Median_width=("width", "median"),
        Median_height=("height", "median"),
        Distinct_resolutions=("resolution", "nunique"),
        Median_aspect=("aspect_ratio", "median"),
    )
    .reindex(cfg.CLASS_ORDER)
    .round(3)
)
resolution_by_class.index = [cfg.CLASS_DISPLAY[n] for n in resolution_by_class.index]

resolution_by_class

# 5. Level 2 — Exact Duplication

SHA-256 answers exactly one question: are these two files identical byte for byte?
It is cheap, exact and complete for that question, and it is the correct first
filter. It is also easy to over-trust: re-saving an image at a different JPEG
quality, or altering a single metadata field, changes the digest entirely while the
picture stays the same. Everything the hash misses is handled at Levels 3 and 4.

Where a hash group spans more than one class folder, the same file has been filed
under two different labels. That is a labelling conflict rather than a duplication
problem, and it is reported separately because it cannot be resolved by deletion.

In [ ]:
exact_groups = va.exact_duplicate_groups(inventory)

retained = {
    va.choose_representative(group)
    for _, group in exact_groups.groupby("sha256")
}
exact_redundant = sorted(set(exact_groups["image_id"]) - retained)

cross_class_hashes = (
    exact_groups.groupby("sha256")["class_name"].nunique().gt(1).sum()
)

print(f"Byte-identical groups          : {exact_groups['sha256'].nunique()}")
print(f"Images inside those groups     : {len(exact_groups)}")
print(f"Redundant copies removed       : {len(exact_redundant)}")
print(f"Groups spanning two classes    : {cross_class_hashes}")

working = (
    inventory[~inventory["image_id"].isin(exact_redundant)]
    .reset_index(drop=True)
)
print(f"Images carried into Level 3    : {len(working)}")

# 6. Convolutional Representation

Each surviving image is passed through an ImageNet-pretrained ResNet-18 (He *et al.*,
2016; Deng *et al.*, 2009) exactly once, and four representations are read out of
that single forward pass using forward hooks:

| Read-out | Shape per image | What it encodes |
|---|---|---|
| `layer4` after global average pooling | 512 | Semantic content, no spatial information |
| `layer2` pooled to a 7 x 7 grid | 128 x 7 x 7 | Mid-level structure, per location |
| `layer3` pooled to a 7 x 7 grid | 256 x 7 x 7 | Higher visual structure, per location |

The network is used only as a fixed feature extractor; nothing is trained here, and
no VTID2 label is seen by it. Each spatial location is L2-normalised across its
channels, so the inner product of two corresponding locations is directly a cosine
similarity in [-1, 1] and no further scaling is needed downstream.

Two practical points matter for reproducing this on modest hardware. The feature
maps are stored in half precision on the CPU — approximately 180 MB for the whole
dataset — and moved to the GPU only in batches, which keeps the analysis inside
4 GB of video memory. And the representations are cached to disk, so every later
experiment is linear algebra over stored arrays rather than repeated inference. The
exploratory version of this analysis re-ran inference for each new idea; caching is
what makes an iterative calibration cheap enough to do properly.

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(cfg.IMAGENET_MEAN, cfg.IMAGENET_STD),
])


class ImagePathDataset(Dataset):
    """Serves preprocessed tensors for a fixed, ordered list of image paths."""

    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))


print(f"Preprocessing: resize to {cfg.IMAGE_SIZE}x{cfg.IMAGE_SIZE}, ImageNet normalisation")

In [ ]:
def extract_representations(paths):
    """Run one forward pass and read layer2, layer3 and layer4 via hooks."""

    backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    backbone.eval().to(device)

    activations = {}

    def make_hook(name):
        def hook(module, inputs, output):
            activations[name] = output.detach()
        return hook

    handles = [
        getattr(backbone, name).register_forward_hook(make_hook(name))
        for name in cfg.FEATURE_LAYERS
    ]

    loader = DataLoader(
        ImagePathDataset(paths, preprocess),
        batch_size=cfg.EXTRACTION_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    grid = (cfg.SPATIAL_GRID, cfg.SPATIAL_GRID)
    deep, spatial2, spatial3 = [], [], []

    with torch.no_grad():
        for batch in loader:
            backbone(batch.to(device, non_blocking=True))

            pooled = F.adaptive_avg_pool2d(activations["layer4"], 1).flatten(1)
            deep.append(F.normalize(pooled, dim=1).cpu())

            for source, store in (("layer2", spatial2), ("layer3", spatial3)):
                maps = F.adaptive_avg_pool2d(activations[source], grid)
                store.append(F.normalize(maps, dim=1).half().cpu())

    for handle in handles:
        handle.remove()

    return (
        torch.cat(deep).numpy().astype(np.float32),
        torch.cat(spatial2),
        torch.cat(spatial3),
    )


cache_file = cfg.CACHE_DIR / "resnet18_representations.npz"

if cache_file.exists():
    stored = np.load(cache_file)
    deep_embeddings = stored["deep"]
    maps_layer2 = torch.from_numpy(stored["layer2"])
    maps_layer3 = torch.from_numpy(stored["layer3"])
    print(f"Loaded cached representations from {cache_file.name}")
else:
    deep_embeddings, maps_layer2, maps_layer3 = extract_representations(working["path"])
    np.savez_compressed(
        cache_file,
        deep=deep_embeddings,
        layer2=maps_layer2.numpy(),
        layer3=maps_layer3.numpy(),
    )
    print(f"Extracted and cached representations to {cache_file.name}")

print(f"Deep embedding : {deep_embeddings.shape}")
print(f"layer2 maps    : {tuple(maps_layer2.shape)}")
print(f"layer3 maps    : {tuple(maps_layer3.shape)}")

# 7. Level 3 — Candidate Generation

Comparing every pair of images with the full spatial descriptor is unnecessary:
the overwhelming majority of pairs are obviously unrelated. The deep embedding is
therefore used as a blocking key. Its known weakness — that it groups *similar*
vehicles rather than *identical* photographs — is precisely what makes it a safe
filter, because it over-includes rather than excludes.

The threshold is set low deliberately. A pair discarded here is invisible to every
later stage, so the only property required of this stage is that it keeps all true
near-duplicates; the recall of this assumption is verified in Section 9.2 against
the labelled reference pairs.

In [ ]:
def generate_candidates(embeddings, threshold, block_size=512):
    """Return the upper-triangular pairs whose deep cosine reaches `threshold`."""

    features = torch.from_numpy(embeddings)
    n_images = len(features)
    columns_index = torch.arange(n_images)

    rows, cols, scores = [], [], []

    for start in range(0, n_images, block_size):
        stop = min(start + block_size, n_images)
        similarity = features[start:stop] @ features.T

        row_index = torch.arange(start, stop)
        upper = columns_index[None, :] > row_index[:, None]
        selected = (similarity >= threshold) & upper

        r, c = selected.nonzero(as_tuple=True)
        rows.append(row_index[r])
        cols.append(c)
        scores.append(similarity[r, c])

    return (
        torch.cat(rows).numpy(),
        torch.cat(cols).numpy(),
        torch.cat(scores).numpy(),
    )


candidate_rows, candidate_cols, candidate_deep = generate_candidates(
    deep_embeddings, cfg.BLOCKING_COSINE
)

total_pairs = len(working) * (len(working) - 1) // 2
print(f"All possible pairs      : {total_pairs:,}")
print(f"Candidate pairs kept    : {len(candidate_rows):,}")
print(f"Retained fraction       : {100 * len(candidate_rows) / total_pairs:.4f}%")
print(f"Images involved         : {len(np.unique(np.concatenate([candidate_rows, candidate_cols]))):,}")

retained_fraction = len(candidate_rows) / total_pairs
if retained_fraction > 0.05:
    print(
        f"\nWARNING: blocking retained {retained_fraction:.1%} of all pairs. "
        f"A threshold of {cfg.BLOCKING_COSINE} is not selective on this "
        "representation, so Level 4 will be slow and its descriptors will be "
        "dominated by unrelated pairs. Raise cfg.BLOCKING_COSINE and confirm "
        "in Section 9.2 that no labelled same-source pair is lost."
    )

# 8. Level 4 — Pair Descriptors from Feature Maps

Each candidate pair is described by six numbers computed from its `layer2` and
`layer3` feature maps. All six are elementary operations on the feature maps
themselves, and each corresponds to a specific claim about what a shared source
would imply.

For a pair of images, let $s_{\ell}$ be the map of local cosine similarities at the
49 spatial locations of layer $\ell$, and $d_{\ell} = 1 - s_{\ell}$ the corresponding
differences.

| Descriptor | Definition | Claim it tests |
|---|---|---|
| `mean` | $\overline{s_{\ell}}$ | The images agree on average |
| `agreement` | $\frac{1}{49}\left|\{u : s_{\ell}(u) \ge 0.90\}\right|$ | The images agree at *most individual locations* |
| `concentration` | $\dfrac{\text{sum of the 5 largest } d_{\ell}}{\sum d_{\ell}}$ | Whatever disagreement exists is *localised* |

The third descriptor is the one that matches the structure of this dataset. Several
VTID2 images appear to be anonymised variants of one another, in which a registration
plate or a face has been obscured. Such a pair is identical almost everywhere and
strongly different in a small region — high agreement, high concentration. Two
photographs of different but similar vehicles disagree moderately across many
locations — lower agreement, low concentration. A mean similarity cannot distinguish
those two situations, because both can produce the same average.

This is also why the earlier exploratory analysis stalled after averaging the layers
together: an average over layers, on top of an average over locations, removes
exactly the spatial information the decision depends on.

In [ ]:
def pair_descriptors(feature_maps, rows, cols, batch_size, agreement_level=0.90, top_k=5):
    """Mean similarity, spatial agreement and difference localisation per pair."""

    means, agreements, concentrations = [], [], []

    for start in range(0, len(rows), batch_size):
        stop = start + batch_size
        left = feature_maps[rows[start:stop]].float().to(device)
        right = feature_maps[cols[start:stop]].float().to(device)

        # Channels are already L2-normalised per location, so the channel-wise
        # inner product at each location is a cosine similarity.
        similarity = (left * right).sum(dim=1).flatten(1)
        difference = (1.0 - similarity).clamp(min=0.0)

        means.append(similarity.mean(dim=1).cpu())
        agreements.append((similarity >= agreement_level).float().mean(dim=1).cpu())

        strongest = difference.topk(top_k, dim=1).values.sum(dim=1)
        concentrations.append((strongest / (difference.sum(dim=1) + 1e-8)).cpu())

    return (
        torch.cat(means).numpy(),
        torch.cat(agreements).numpy(),
        torch.cat(concentrations).numpy(),
    )


l2_mean, l2_agreement, l2_concentration = pair_descriptors(
    maps_layer2, candidate_rows, candidate_cols, cfg.PAIR_BATCH_SIZE
)
l3_mean, l3_agreement, l3_concentration = pair_descriptors(
    maps_layer3, candidate_rows, candidate_cols, cfg.PAIR_BATCH_SIZE
)

candidates = pd.DataFrame({
    "index_a": candidate_rows,
    "index_b": candidate_cols,
    "image_a": working["image_id"].to_numpy()[candidate_rows],
    "image_b": working["image_id"].to_numpy()[candidate_cols],
    "deep_cosine": candidate_deep,
    "l2_mean": l2_mean,
    "l2_agreement": l2_agreement,
    "l2_concentration": l2_concentration,
    "l3_mean": l3_mean,
    "l3_agreement": l3_agreement,
    "l3_concentration": l3_concentration,
})

DESCRIPTORS = [
    "deep_cosine",
    "l2_mean", "l2_agreement", "l2_concentration",
    "l3_mean", "l3_agreement", "l3_concentration",
]

candidates.to_csv(cfg.RESULT_DIR / "candidate_pairs.csv", index=False)
candidates[DESCRIPTORS].describe().T.round(4)

# 9. Calibrating the Deletion Rule

## 9.1 A reference set of manually labelled pairs

The descriptors are numbers; only a human can say whether two photographs share a
source. A reference set of manually labelled pairs is therefore required, and the
way it is sampled determines how much labelling effort is needed.

Labelling candidate pairs in the order they happen to appear spends almost all of
the effort on cases that any rule would classify correctly. Instead, pairs are
sampled in strata spanning the range of `l2_agreement`, which concentrates the
labels near the decision boundary where they carry information. Roughly eighty
labelled pairs obtained this way constrain the rule better than several hundred
labelled sequentially.

**Labelling protocol.** A pair is labelled `1` only when the two images are
evidently the same underlying photograph — including re-encoded, re-cropped,
rescaled or partially obscured variants. A pair showing the same vehicle model, or
even the same vehicle in a different frame, is labelled `0`. The distinction is
*shared source image*, not *shared subject*, and it is applied consistently because
it is the property that makes an evaluation optimistic.

Running the next cell writes a labelling template and the corresponding contact
sheets to disk. The completed file is saved as `data/labels/pair_labels.csv` with
columns `image_a`, `image_b`, `target`.

In [ ]:
def build_labelling_template(frame, n_strata=8, per_stratum=10, seed=cfg.RANDOM_SEED):
    """Sample candidate pairs across the descriptor range for manual labelling."""

    rng = np.random.default_rng(seed)
    edges = np.linspace(
        frame["l2_agreement"].min(), frame["l2_agreement"].max(), n_strata + 1
    )

    picks = []
    for low, high in zip(edges[:-1], edges[1:]):
        stratum = frame[(frame["l2_agreement"] >= low) & (frame["l2_agreement"] < high)]
        if len(stratum) == 0:
            continue
        take = min(per_stratum, len(stratum))
        picks.append(stratum.iloc[rng.choice(len(stratum), take, replace=False)])

    return pd.concat(picks).sort_values("l2_agreement", ascending=False).reset_index(drop=True)


if not cfg.PAIR_LABEL_FILE.exists():
    template = build_labelling_template(candidates)
    template = template.assign(target="")

    template_path = cfg.LABEL_DIR / "pair_labels_template.csv"
    template.to_csv(template_path, index=False)

    sheet_dir = cfg.FIGURE_DIR / "labelling"
    sheet_dir.mkdir(parents=True, exist_ok=True)
    for page, start in enumerate(range(0, len(template), 8), start=1):
        block = template.iloc[start:start + 8]
        figure = va.contact_sheet(
            list(zip(block["index_a"], block["index_b"])),
            working,
            block["l2_agreement"].to_numpy(),
            title=f"Labelling page {page} — layer2 spatial agreement",
        )
        va.save_figure(figure, f"labelling/page_{page:02d}", cfg.FIGURE_DIR)
        plt.close(figure)

    print(f"Wrote {len(template)} pairs to {template_path}")
    print(f"Contact sheets written to {sheet_dir}")
else:
    print(f"Using existing labels: {cfg.PAIR_LABEL_FILE}")

In [ ]:
def pair_key(frame):
    """Order-independent identifier for a pair, so a hand-written label file
    may list either image first."""
    return [
        "|".join(sorted((a, b)))
        for a, b in zip(frame["image_a"], frame["image_b"])
    ]


candidates["pair_key"] = pair_key(candidates)

if cfg.PAIR_LABEL_FILE.exists():
    labelled_raw = pd.read_csv(cfg.PAIR_LABEL_FILE).dropna(subset=["target"])
    labelled_raw["pair_key"] = pair_key(labelled_raw)

    labelled = (
        candidates.merge(
            labelled_raw[["pair_key", "target"]].drop_duplicates("pair_key"),
            on="pair_key",
            how="inner",
        )
        .astype({"target": int})
    )

    unmatched = len(labelled_raw) - len(labelled)
    if unmatched:
        print(f"Note: {unmatched} labelled pairs are not candidate pairs and were ignored.")
        print("      Expected when a labelled image was an exact duplicate removed at")
        print("      Level 2. Any other case means blocking discarded a labelled pair,")
        print("      which no later stage can recover and should be investigated.")

    is_calibrated = len(labelled) > 0
else:
    labelled = candidates.iloc[:0].assign(target=pd.Series(dtype=int))
    is_calibrated = False

if is_calibrated:
    print(f"Labelled pairs        : {len(labelled)}")
    print(f"Same source (1)       : {int(labelled['target'].sum())}")
    print(f"Different source (0)  : {int((1 - labelled['target']).sum())}")
else:
    print("No labels available yet — Section 9.3 will fall back to cfg.FALLBACK_RULE.")

## 9.2 Which descriptors can decide identity?

Before any threshold is chosen, each descriptor is assessed on its own. Two figures
are reported for each: the area under the ROC curve, which measures how well the
descriptor ranks pairs, and the highest recall attainable at perfect precision,
which is the only quantity the deletion decision actually depends on.

A descriptor with a high AUC but zero recall at perfect precision is useless here:
it orders pairs sensibly, yet has no threshold at which deletion is safe. That is
the expected outcome for `deep_cosine`, and stating it as a number replaces the many
pages of visual comparison that the exploratory analysis spent reaching the same
conclusion.

In [ ]:
from sklearn.metrics import roc_auc_score


def max_recall_at_full_precision(scores, targets):
    """Highest recall achievable by a one-sided threshold with no false positives."""

    order = np.argsort(-scores, kind="stable")
    ordered = targets[order]

    recovered = 0
    for label in ordered:
        if label == 0:
            break
        recovered += 1

    positives = int(targets.sum())
    return recovered / positives if positives else np.nan


if is_calibrated:
    targets = labelled["target"].to_numpy()

    separability = pd.DataFrame([
        {
            "Descriptor": name,
            "AUC": roc_auc_score(targets, labelled[name]),
            "Recall at precision 1.00": max_recall_at_full_precision(
                labelled[name].to_numpy(), targets
            ),
            "Positive median": labelled.loc[labelled["target"] == 1, name].median(),
            "Negative median": labelled.loc[labelled["target"] == 0, name].median(),
        }
        for name in DESCRIPTORS
    ]).set_index("Descriptor").round(4)

    display(separability)
    separability.to_csv(cfg.RESULT_DIR / "descriptor_separability.csv")
else:
    print("Skipped: requires the labelled reference set.")

## 9.3 Selecting the operating point

The deletion rule is a conjunction of three one-sided conditions:

$$\text{delete-eligible} \iff (a_2 \ge \alpha) \wedge (m_3 \ge \beta) \wedge (c_2 \ge \gamma)$$

where $a_2$ is `l2_agreement`, $m_3$ is `l3_mean` and $c_2$ is `l2_concentration`.
The three conditions are not redundant: the first requires broad agreement, the
second requires that agreement to persist at a more abstract level of the network,
and the third requires the residual disagreement to be localised.

The operating point is chosen by an explicit, stated objective rather than by
inspection:

> maximise recall subject to precision = 1.00 on the labelled reference set.

Precision is constrained rather than traded off because deletion is irreversible,
while the cost of a missed duplicate is bounded — Section 10 routes every rejected
candidate into the leakage-risk graph, where it still constrains the partition.
Reporting the full trade-off, rather than only the selected point, is what makes
the choice auditable.

In [ ]:
def evaluate_rule(frame, targets, alpha, beta, gamma):
    predicted = (
        (frame["l2_agreement"].to_numpy() >= alpha)
        & (frame["l3_mean"].to_numpy() >= beta)
        & (frame["l2_concentration"].to_numpy() >= gamma)
    )
    true_positive = int((predicted & (targets == 1)).sum())
    false_positive = int((predicted & (targets == 0)).sum())
    positives = int(targets.sum())

    precision = true_positive / (true_positive + false_positive) if predicted.any() else np.nan
    recall = true_positive / positives if positives else np.nan
    return precision, recall, true_positive, false_positive


if is_calibrated:
    alpha_grid = np.round(np.arange(0.30, 0.96, 0.05), 2)
    beta_grid = np.round(np.arange(0.70, 0.99, 0.02), 2)
    gamma_grid = np.round(np.arange(0.20, 0.71, 0.05), 2)

    searched = pd.DataFrame(
        [
            dict(zip(
                ["alpha", "beta", "gamma", "precision", "recall", "tp", "fp"],
                (a, b, g) + evaluate_rule(labelled, targets, a, b, g),
            ))
            for a in alpha_grid for b in beta_grid for g in gamma_grid
        ]
    ).dropna(subset=["precision"])

    safe = searched[(searched["precision"] >= 1.0) & (searched["recall"] > 0)]

    if len(safe) == 0:
        # No conjunction separates the labelled pairs without error. This is a
        # finding about the descriptors, not a failure of the search, and it is
        # reported rather than worked around: the fallback maximises precision,
        # and the resulting rule must not be used for deletion until either the
        # reference set or the descriptor set has been revisited.
        attainable = searched["precision"].max()
        safe = searched[searched["precision"] >= attainable - 1e-9]
        print("WARNING: no rule achieved precision 1.00 on the labelled pairs.")
        print(f"         Best attainable precision is {attainable:.3f}.")
        print("         Treat the selected rule as provisional (see Section 15).")

    best = safe.sort_values(
        ["recall", "alpha", "beta", "gamma"], ascending=[False, True, True, True]
    ).iloc[0]

    rule = {
        "l2_agree_090": float(best["alpha"]),
        "l3_mean": float(best["beta"]),
        "l2_concentration": float(best["gamma"]),
    }

    print(f"Rules evaluated                : {len(searched):,}")
    print(f"Rules with precision = 1.00    : {len(safe):,}")
    print(f"Selected  l2_agreement      >= {rule['l2_agree_090']:.2f}")
    print(f"Selected  l3_mean           >= {rule['l3_mean']:.2f}")
    print(f"Selected  l2_concentration  >= {rule['l2_concentration']:.2f}")
    print(f"Reference-set recall           : {best['recall']:.3f} "
          f"({int(best['tp'])}/{int(targets.sum())} same-source pairs recovered)")
else:
    rule = dict(cfg.FALLBACK_RULE)
    print("Falling back to cfg.FALLBACK_RULE (uncalibrated):")
    print(rule)

In [ ]:
if is_calibrated:
    trade_off = (
        searched.groupby("recall")
        .agg(best_precision=("precision", "max"))
        .reset_index()
        .sort_values("recall")
    )

    figure, axis = plt.subplots(figsize=(5.5, 3.6))
    axis.plot(trade_off["recall"], trade_off["best_precision"], marker="o", markersize=3)
    axis.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)
    axis.axvline(best["recall"], color="#C44E52", linestyle=":", linewidth=1.0)
    axis.set_xlabel("Recall on the labelled reference set")
    axis.set_ylabel("Best attainable precision")
    axis.set_title("Deletion rule: attainable precision by recall")
    figure.tight_layout()

    va.save_figure(figure, "02_deletion_rule_trade_off", cfg.FIGURE_DIR)
    plt.show()

## 9.4 Robustness of the selected rule

A rule fitted to fewer than a hundred labels can encode the labels rather than the
phenomenon. Two independent checks are applied.

First, a depth-two decision tree is fitted to all six descriptors under
leave-one-out cross-validation. If a learned classifier with access to the same
information does not outperform the hand-specified conjunction, the conjunction is
not leaving usable signal behind, and its simplicity is then an advantage: it can be
stated in one line of a report and defended in a viva.

Second, the rule is applied to the labelled pairs' immediate neighbourhood in
descriptor space by perturbing each threshold. A rule whose behaviour changes
sharply under a small perturbation has been fitted to individual labels.

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score

if is_calibrated and labelled["target"].nunique() == 2:
    features = labelled[DESCRIPTORS].to_numpy()

    tree = DecisionTreeClassifier(max_depth=2, random_state=cfg.RANDOM_SEED)
    predicted = cross_val_predict(tree, features, targets, cv=LeaveOneOut())

    rule_precision, rule_recall, _, _ = evaluate_rule(
        labelled, targets,
        rule["l2_agree_090"], rule["l3_mean"], rule["l2_concentration"],
    )

    comparison = pd.DataFrame([
        {
            "Method": "Selected conjunction",
            "Precision": rule_precision,
            "Recall": rule_recall,
        },
        {
            "Method": "Depth-2 decision tree (leave-one-out CV)",
            "Precision": precision_score(targets, predicted, zero_division=0),
            "Recall": recall_score(targets, predicted, zero_division=0),
        },
    ]).set_index("Method").round(4)

    display(comparison)

    perturbation = pd.DataFrame([
        {
            "Perturbation": f"{delta:+.2f} on every threshold",
            **dict(zip(
                ["Precision", "Recall", "TP", "FP"],
                evaluate_rule(
                    labelled, targets,
                    rule["l2_agree_090"] + delta,
                    rule["l3_mean"] + delta,
                    rule["l2_concentration"] + delta,
                ),
            )),
        }
        for delta in (-0.04, -0.02, 0.0, 0.02, 0.04)
    ]).set_index("Perturbation").round(4)

    display(perturbation)
else:
    print("Skipped: requires a labelled reference set containing both classes.")

In [ ]:
accepted_mask = (
    (candidates["l2_agreement"] >= rule["l2_agree_090"])
    & (candidates["l3_mean"] >= rule["l3_mean"])
    & (candidates["l2_concentration"] >= rule["l2_concentration"])
).to_numpy()

ambiguous_mask = (~accepted_mask) & (
    (candidates["l2_agreement"] >= cfg.AMBIGUOUS_RULE["l2_agree_090"])
    & (candidates["l3_mean"] >= cfg.AMBIGUOUS_RULE["l3_mean"])
).to_numpy()

accepted_edges = candidates.loc[accepted_mask, ["index_a", "index_b"]].to_numpy()
ambiguous_edges = candidates.loc[ambiguous_mask, ["index_a", "index_b"]].to_numpy()
ambiguous_scores = candidates.loc[ambiguous_mask, "l2_agreement"].to_numpy()

print(f"Candidate pairs        : {len(candidates):,}")
print(f"Accepted for deletion  : {accepted_mask.sum():,}")
print(f"Ambiguous, retained    : {ambiguous_mask.sum():,}")
print(f"Rejected outright      : {(~accepted_mask & ~ambiguous_mask).sum():,}")

In [ ]:
weakest = (
    candidates[accepted_mask]
    .sort_values("l2_agreement")
    .head(8)
)

figure = va.contact_sheet(
    list(zip(weakest["index_a"], weakest["index_b"])),
    working,
    weakest["l2_agreement"].to_numpy(),
    title="Weakest accepted pairs (layer2 spatial agreement shown)",
)
va.save_figure(figure, "03_weakest_accepted_pairs", cfg.FIGURE_DIR)
plt.show()

The figure above is the only pairwise visual evidence retained in this notebook, and
it shows the eight *weakest* pairs the rule accepts. If the marginal cases are
convincing, the confident ones need not be displayed. This is a deliberate reversal
of the exploratory approach, which rendered many grids of comparable pairs and
inflated the notebook to a size that could not be reviewed or submitted; the
remaining sheets are written to `figures/labelling/` as files.

# 10. From Pairs to Groups

## 10.1 Why pairs cannot be deleted directly

Deleting one image from every accepted pair is wrong for two reasons. It counts
relationships rather than images — a set of four copies produces six pairs — and it
gives no consistent answer about which copy to keep. Pairs must first become groups.

The obvious construction, taking connected components of the accepted pairs, is
also unsafe. Similarity is not transitive: if A resembles B and B resembles C,
nothing guarantees that A resembles C. A chain of individually plausible links can
therefore merge unrelated images into one component, and a single such component can
absorb a large share of the dataset.

## 10.2 Full pairwise consistency

A component is treated as a duplicate group only when *every* pair inside it was
independently accepted — that is, when the component is a complete graph. This makes
the deletion decision as conservative as the pairwise rule that produced it, with no
reliance on transitivity.

Components that are connected but not complete are not deleted from. They are
retained in full and passed to Section 10.3 as evidence of possible shared sources.
This is the point where the two decisions of Section 1.1 separate in the code.

In [ ]:
component_labels = va.connected_components(len(working), accepted_edges)
consistent_groups, chained_groups = va.split_components_by_consistency(
    component_labels, accepted_edges
)

print(f"Connected components (size > 1)     : {len(consistent_groups) + len(chained_groups)}")
print(f"Fully consistent duplicate groups   : {len(consistent_groups)}")
print(f"Chained, not fully consistent       : {len(chained_groups)}")

if consistent_groups:
    sizes = pd.Series([len(g) for g in consistent_groups])
    print(f"Images inside consistent groups     : {int(sizes.sum())}")
    print(f"Group size — median {sizes.median():.0f}, max {sizes.max()}")

In [ ]:
group_frames = [working.iloc[nodes] for nodes in consistent_groups]

cross_class_groups = [
    frame for frame in group_frames if frame["class_name"].nunique() > 1
]

print(f"Duplicate groups spanning two or more classes: {len(cross_class_groups)}")

if cross_class_groups:
    conflict_report = pd.concat(
        frame.assign(conflict_id=index)[["conflict_id", "image_id", "class_name"]]
        for index, frame in enumerate(cross_class_groups)
    )
    conflict_report.to_csv(cfg.RESULT_DIR / "label_conflicts.csv", index=False)
    display(conflict_report.head(20))

A duplicate group whose members carry different class labels is a labelling
conflict, not a duplication problem: the same photograph has been filed under two
vehicle types, so at least one label is wrong. Deleting the redundant copies would
silently keep whichever label survived and hide the error. These groups are exported
to `results/label_conflicts.csv` for manual review, and are excluded from automatic
deletion. Any that remain unresolved are still held together during partitioning, so
they cannot inflate the reported accuracy.

In [ ]:
deletable = [
    frame for frame in group_frames if frame["class_name"].nunique() == 1
]

redundant_images = []
for frame in deletable:
    keeper = va.choose_representative(frame)
    redundant_images.extend(sorted(set(frame["image_id"]) - {keeper}))

clean = working[~working["image_id"].isin(redundant_images)].reset_index(drop=True)

print(f"High-confidence duplicate groups   : {len(deletable)}")
print(f"Redundant images removed           : {len(redundant_images)}")
print(f"Final dataset size                 : {len(clean)}")

## 10.3 Leakage-risk groups

Every relation that was *not* strong enough to justify deletion is now reused for a
weaker purpose. The leakage-risk graph contains:

- the accepted pairs, including those inside chained components and label-conflict
  groups, none of which were deleted from;
- the ambiguous pairs, which fell below the deletion rule but above a permissive
  floor.

Ambiguous relations are filtered by a mutual nearest-neighbour condition before use.
Without it, a single generic image that is moderately similar to many others acts as
a hub and chains large parts of the dataset into one component; if such a component
grows, a group-disjoint partition with the required class proportions becomes
impossible to construct. Requiring that each image ranks the other among its closest
few neighbours keeps reciprocal relations and discards hub effects.

In [ ]:
mutual_ambiguous = va.mutual_knn_edges(
    ambiguous_edges, ambiguous_scores, k=cfg.MUTUAL_KNN_K
)

print(f"Ambiguous relations              : {len(ambiguous_edges):,}")
print(f"Surviving mutual {cfg.MUTUAL_KNN_K}-NN filter       : {len(mutual_ambiguous):,}")

# Relations are indexed against `working`; the partition is built on `clean`.
# Both relation sets are re-indexed onto `clean` so that every later check
# operates in a single, consistent index space.
position = {image: index for index, image in enumerate(clean["image_id"])}
source_ids = working["image_id"].to_numpy()


def reindex_onto_clean(edges):
    kept = [
        (position[source_ids[a]], position[source_ids[b]])
        for a, b in edges
        if source_ids[a] in position and source_ids[b] in position
    ]
    return np.array(kept, dtype=int) if kept else np.empty((0, 2), dtype=int)


accepted_relations = reindex_onto_clean(accepted_edges)
ambiguous_relations = reindex_onto_clean(mutual_ambiguous)

risk_edges = (
    np.vstack([accepted_relations, ambiguous_relations])
    if len(ambiguous_relations) else accepted_relations
)

clean["group_id"] = va.connected_components(len(clean), risk_edges)

group_sizes = clean["group_id"].value_counts()
largest_fraction = group_sizes.max() / len(clean)

print(f"Leakage-risk relations retained  : {len(risk_edges):,}")
print(f"Groups (an isolated image is a group of one): {clean['group_id'].nunique():,}")
print(f"Groups with more than one image  : {int((group_sizes > 1).sum()):,}")
print(f"Largest group                    : {int(group_sizes.max())} images "
      f"({largest_fraction:.1%} of the dataset)")

print("\nGroup size distribution")
print(group_sizes.value_counts().sort_index().rename("Groups").head(10).to_string())

In [ ]:
if largest_fraction > cfg.MAX_GROUP_FRACTION:
    raise RuntimeError(
        f"The largest leakage-risk group holds {largest_fraction:.1%} of the dataset "
        f"({int(group_sizes.max())} of {len(clean)} images), above the "
        f"{cfg.MAX_GROUP_FRACTION:.0%} limit in cfg.MAX_GROUP_FRACTION.\n\n"
        "A group this large cannot be split, so a group-disjoint partition with the "
        "target class proportions does not exist. Chaining, not duplication, is the "
        "usual cause. In order of likelihood:\n"
        "  1. cfg.AMBIGUOUS_RULE admits too many weak relations;\n"
        "  2. cfg.MUTUAL_KNN_K is too large, so hub images still bridge components;\n"
        "  3. the deletion rule of Section 9.3 is too permissive, which the "
        "labelled reference set should have caught.\n\n"
        "Tighten these and re-run from Section 9.4. Do not raise the limit to "
        "proceed: it would produce a partition whose groups are not meaningful."
    )

print(f"Largest group is {largest_fraction:.1%} of the dataset "
      f"(limit {cfg.MAX_GROUP_FRACTION:.0%}) — groups are usable for partitioning.")

## 10.4 Checking that the groups remain usable

Grouping can fail in a way that is silent until the partition is built. If chaining
merges a large share of the dataset into one component, that component must be
assigned to a single partition, and the requested proportions become impossible to
achieve. The result is not an error message but a partition that is quietly unusable —
in the extreme, one in which the validation and test sets are empty.

The check below makes that failure loud and immediate, and states the cause. It is
a diagnostic on the *rules*, not on the data: a giant component means the deletion
rule or the ambiguous floor is too permissive, or the mutual nearest-neighbour
constraint is too weak.

# 11. Leakage-Controlled Partitioning

The unit of partitioning is the group, not the image. Every image that might share a
source with another travels with it, so no source photograph can appear on both
sides of the evaluation boundary. This is group-aware splitting in the standard
sense; the contribution of the preceding sections is the construction of defensible
groups for a dataset that carries no vehicle or sequence identifier.

Groups are atomic and their sizes are unequal, so the target 70 / 15 / 15 proportions
cannot generally be met exactly. Groups are assigned largest first to whichever
partition is furthest below its quota for the group's dominant class, which keeps the
realised proportions close to the target while preserving group integrity. Placing
the large groups first matters: leaving them until the end would force a large block
into whichever partition remained, distorting the class balance.

In [ ]:
clean["split"] = va.group_stratified_split(
    clean, cfg.SPLIT_RATIOS, seed=cfg.RANDOM_SEED
)

composition = va.split_composition(clean, cfg.CLASS_DISPLAY)
composition

In [ ]:
partition_counts = (
    clean["split"].value_counts()
    .reindex(["train", "val", "test"])
    .fillna(0)
    .astype(int)
)

realised = pd.DataFrame({
    "Images": partition_counts,
    "Realised (%)": (100 * partition_counts / partition_counts.sum()).round(2),
    "Target (%)": [100 * cfg.SPLIT_RATIOS[s] for s in ["train", "val", "test"]],
})
realised.index = ["Train", "Validation", "Test"]

display(realised.round(2))

class_shares = (
    (pd.crosstab(clean["class_name"], clean["split"], normalize="columns") * 100)
    .reindex(index=cfg.CLASS_ORDER, columns=["train", "val", "test"])
    .fillna(0)
    .round(2)
)
class_shares.index = [cfg.CLASS_DISPLAY[name] for name in class_shares.index]

print("\nClass composition of each partition (%)")
display(class_shares)

In [ ]:
figure, axis = plt.subplots(figsize=(6.5, 3.6))
class_shares.T.plot(kind="bar", stacked=True, ax=axis, width=0.6)
axis.set_ylabel("Share of partition (%)")
axis.set_xlabel("")
axis.set_title("Class composition is preserved across partitions")
axis.tick_params(axis="x", rotation=0)
axis.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
figure.tight_layout()

va.save_figure(figure, "04_partition_composition", cfg.FIGURE_DIR)
plt.show()

# 12. Verification

The partition is now checked against the objects that produced it being set aside.
Each quantity below is recomputed from the final assignment, so an error anywhere in
the pipeline would appear here rather than be reproduced by construction.

The first three checks are discrete and must be zero. The fourth is different in
kind: it reports the most similar image pair that still crosses each partition
boundary. No audit can prove that every near-duplicate was found, and a single
number stating the worst residual case is a more honest claim than silence.

In [ ]:
verification = va.verify_partition(clean, accepted_relations, ambiguous_relations)
display(verification)

assert verification["Count (must be 0)"].sum() == 0, "Leakage detected in the partition"
print("All discrete leakage checks passed.")

In [ ]:
clean_positions = [
    index for index, image in enumerate(working["image_id"]) if image in position
]
clean_embeddings = deep_embeddings[np.array(clean_positions)]

residual = va.max_cross_split_similarity(
    clean_embeddings, clean["split"].to_numpy()
)
residual.round(4)

# 13. Audit Summary

In [ ]:
spanning_groups = int(
    clean.groupby("group_id")["split"].nunique().gt(1).sum()
)

summary = pd.DataFrame([
    ("Source images", len(inventory)),
    ("Unreadable or truncated files", int((~inventory["readable"]).sum())),
    ("Exact duplicate groups (SHA-256)", exact_groups["sha256"].nunique()),
    ("Exact redundant images removed", len(exact_redundant)),
    ("Images entering CNN analysis", len(working)),
    ("Candidate pairs after blocking", len(candidates)),
    ("Pairs accepted as same-source", int(accepted_mask.sum())),
    ("High-confidence duplicate groups", len(deletable)),
    ("Near-duplicate redundant images removed", len(redundant_images)),
    ("Label-conflict groups referred for review", len(cross_class_groups)),
    ("Final dataset", len(clean)),
    ("Leakage-risk groups", int(clean["group_id"].nunique())),
    ("Training images", int((clean["split"] == "train").sum())),
    ("Validation images", int((clean["split"] == "val").sum())),
    ("Test images", int((clean["split"] == "test").sum())),
    ("Groups spanning more than one partition", spanning_groups),
], columns=["Audit stage", "Value"])

summary.to_csv(cfg.AUDIT_SUMMARY, index=False)
clean.to_csv(cfg.SPLIT_MANIFEST, index=False)

print(f"Written: {cfg.AUDIT_SUMMARY.name}, {cfg.SPLIT_MANIFEST.name}")
summary

# 14. Reconciliation with the Reference Run

This pipeline replaces an earlier exploratory analysis of the same dataset. That
analysis reached a defensible result but did so through an open-ended search whose
record was too large to submit. The comparison below states, rather than assumes,
how closely the rewritten pipeline reproduces it.

Exact agreement is not expected at every row and is not required. The deletion rule
here is estimated from the labelled reference set instead of being tuned by
inspection, and group-atomic allocation rounds the partition sizes differently
whenever the group structure differs by even one relation. What the comparison must
show is agreement in magnitude at every stage and identical conclusions: the same
order of duplication, the same final dataset size to within a small tolerance, and
zero groups spanning partitions.

In [ ]:
reference = cfg.REFERENCE_RUN
observed = {
    "raw_images": len(inventory),
    "exact_duplicate_redundant": len(exact_redundant),
    "after_exact_dedup": len(working),
    "high_confidence_duplicate_groups": len(deletable),
    "near_duplicate_redundant": len(redundant_images),
    "final_images": len(clean),
    "train_images": int((clean["split"] == "train").sum()),
    "val_images": int((clean["split"] == "val").sum()),
    "test_images": int((clean["split"] == "test").sum()),
    "groups_spanning_splits": spanning_groups,
}

reconciliation = pd.DataFrame({
    "Reference run": pd.Series(reference),
    "This pipeline": pd.Series(observed),
})
reconciliation["Difference"] = (
    reconciliation["This pipeline"] - reconciliation["Reference run"]
)
reconciliation["Relative (%)"] = (
    100 * reconciliation["Difference"] / reconciliation["Reference run"].replace(0, np.nan)
).round(2)

reconciliation

In [ ]:
reference_split = pd.DataFrame(
    cfg.REFERENCE_CLASS_SPLIT, index=["train", "val", "test"]
).T
reference_split.index = [cfg.CLASS_DISPLAY[name] for name in reference_split.index]

observed_split = composition.drop(index="Total").drop(columns="Total")

comparison = observed_split.subtract(reference_split.reindex(observed_split.index))
comparison.columns = [f"{column} (difference)" for column in comparison.columns]

pd.concat([observed_split, comparison], axis=1)

# 15. Limitations

**The reference set is conditional on blocking.** Precision and recall in Section 9
are measured on pairs that survived Level 3. They describe the decision rule, not the
whole pipeline, and they cannot detect a near-duplicate that blocking discarded. The
low blocking threshold and the residual-similarity report in Section 12 bound that
risk but do not eliminate it.

**The reference set is small.** Fewer than a hundred labelled pairs support the
calibration. The leave-one-out and perturbation checks in Section 9.4 test whether
the rule generalises within that set; they cannot rule out a systematic labelling bias
shared by all of them, since one person applied the protocol.

**Group construction is a heuristic, not ground truth.** VTID2 carries no vehicle,
camera, sequence or capture-time identifier. If it did, grouping by that identifier
would be strictly preferable to inferring groups from appearance, and the analysis in
Sections 6 to 10 would be unnecessary. The groups here approximate an unobserved
variable, and the mutual nearest-neighbour parameter influences their extent.

**Deletion is conservative by construction.** The rule is calibrated to make no false
deletions, which guarantees that some genuine duplicates survive. Those survivors are
not lost from the analysis: they remain in the leakage-risk graph and still constrain
the partition. The residual cost is a slightly smaller effective sample, not an
optimistic evaluation.

**The extractor is pretrained on ImageNet.** The representation reflects that
distribution. It is used only to compare images with one another and never to predict
a VTID2 label, so it cannot leak label information into the partition, but its
sensitivity to the vehicle imagery in this dataset is assumed rather than measured.

# References

Barz, B. and Denzler, J. (2020) 'Do we train on test data? Purging CIFAR of
near-duplicates', *Journal of Imaging*, 6(6), 41.

Christen, P. (2012) *Data Matching: Concepts and Techniques for Record Linkage,
Entity Resolution, and Duplicate Detection*. Berlin: Springer.

Deng, J., Dong, W., Socher, R., Li, L.-J., Li, K. and Fei-Fei, L. (2009) 'ImageNet: a
large-scale hierarchical image database', *IEEE Conference on Computer Vision and
Pattern Recognition*, pp. 248-255.

He, K., Zhang, X., Ren, S. and Sun, J. (2016) 'Deep residual learning for image
recognition', *IEEE Conference on Computer Vision and Pattern Recognition*,
pp. 770-778.

Kaufman, S., Rosset, S., Perlich, C. and Stitelman, O. (2012) 'Leakage in data
mining: formulation, detection, and avoidance', *ACM Transactions on Knowledge
Discovery from Data*, 6(4), pp. 1-21.

Paszke, A. *et al.* (2019) 'PyTorch: an imperative style, high-performance deep
learning library', *Advances in Neural Information Processing Systems*, 32.

Pedregosa, F. *et al.* (2011) 'Scikit-learn: machine learning in Python', *Journal of
Machine Learning Research*, 12, pp. 2825-2830.

Recht, B., Roelofs, R., Schmidt, L. and Shankar, V. (2019) 'Do ImageNet classifiers
generalize to ImageNet?', *Proceedings of the 36th International Conference on
Machine Learning*, pp. 5389-5400.

Zeiler, M.D. and Fergus, R. (2014) 'Visualizing and understanding convolutional
networks', *European Conference on Computer Vision*, pp. 818-833.

**Dataset.** VTID2 (Vehicle Type Image Dataset 2). *Insert the full citation for the
copy of the dataset you obtained — originating authors, year, title, repository or
DOI, and date accessed — before submission.*